In [86]:
import sys, os
import numpy as np


In [93]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [94]:
def softmax_nonorm(a):
    exp_a = np.exp(a)
    sum_exp_a = np.sum(exp_a)
    return exp_a / sum_exp_a

def softmax(a): #개선
    c = np.max(a)
    return softmax_nonorm(a-c)

In [95]:
sys.path.insert(0, os.getcwd())

from mnist import load_mnist

In [96]:
(x_train, t_train), (x_test, t_test) = \
    load_mnist(flatten = True, normalize = False)

/Users/jeonghowon/Desktop/군러닝/MitDeep/mnist.py:109: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dataset = pickle.load(f)


In [97]:
import numpy as np
import pickle
from PIL import Image

def img_show(img):
    pil_img = Image.fromarray(np.uint8(img))
    pil_img.show()

img = x_train[0]
label = t_train[0]
print(label)

print(img.shape) #flatten되서

#img_show(img)

img = img.reshape(28, 28)
print(img.shape)


5
(784,)
(28, 28)


In [98]:
def get_data():
    (x_train, t_train), (x_test, t_test) = \
        load_mnist(normalize=True, flatten=True, one_hot_label=False)
    
    return x_test, t_test

def init_network():
    with open("sample_weight.pkl", 'rb') as f:
        network = pickle.load(f)

    return network

def predict(network, x):
    W1, W2, W3 = network['W1'], network['W2'], network['W3']
    b1, b2, b3 = network['b1'], network['b2'], network['b3']

    a1 = np.dot(x, W1) + b1
    z1 = sigmoid(a1)
    a2 = np.dot(z1, W2) + b2
    z2 = sigmoid(a2)
    a3 = np.dot(z2, W3) + b3
    y = softmax(a3)

    return y


In [99]:
x, t = get_data()
network = init_network()

/var/folders/kw/b2hg_vc50cg4rktl0x3hr8tw0000gn/T/ipykernel_68290/1318228235.py:9: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  network = pickle.load(f)


In [ ]:
accuracy_cnt = 0
for i in range(len(x)):
    y = predict(network, x[i])
    p = np.argmax(y) #최대인 값의 idx를 리턴하는 함수
    if p == t[i]:
        accuracy_cnt += 1

print("Accuracy: " + str(float(accuracy_cnt) / len(x)))

Accuracy: 0.9352


정규화와 전처리를 구분:

정규화란 데이터를 특정 범위로 변환하여 통계적 이점을 살림
전처리는 신경망의 입력 데이터에 특정 변환을 가하는 작업

이 장에서 MNIST 데이터에 대한 전처리 작업으로 정규화를 실행하였다.

In [109]:
x, _ = get_data() #x만 보자는거
print(x.shape)
print(x[0].shape)

network = init_network()
print(network['W1'].shape)
print(network['W2'].shape)
print(network['W3'].shape)


(10000, 784)
(784,)
(784, 50)
(50, 100)
(100, 10)


/Users/jeonghowon/Desktop/군러닝/MitDeep/mnist.py:109: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dataset = pickle.load(f)
/var/folders/kw/b2hg_vc50cg4rktl0x3hr8tw0000gn/T/ipykernel_68290/1318228235.py:9: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  network = pickle.load(f)


이때 한개의 이미지가 flatten 되어 입력으로 들어가는 과정이지만, 
더욱 효율적인 병렬 계산을 위해 100 * 784처럼 100개(예)의 이미지의 배치 처리를 활용할 수 있다!

In [133]:
x, t = get_data()
network = init_network()

batch_size = 100
accuracy = 0

for i in range(0, len(x), batch_size): #배치 사이즈 만큼 점프
    x_batch = x[i:i+batch_size] #x[i]부터 x[i+batch_size]꺄지
    y_batch = predict(network, x_batch)
    p = np.argmax(y_batch, axis = 1)
    accuracy_cnt += np.sum(p == t[i:i+batch_size])

    #sum을 사용하는 이유는~?

print("Accuracy:" + str(float(accuracy_cnt) / len(x)))

Accuracy:23.38


/var/folders/kw/b2hg_vc50cg4rktl0x3hr8tw0000gn/T/ipykernel_68290/1318228235.py:9: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  network = pickle.load(f)
